# Tavily Search API 발급 및 사용

## 실습 목표
---
Adaptive RAG를 구현하기 위해 Tavily Search API를 발급하고 LangChain을 통해 사용해 볼 것입니다.

24년 10월 기준 API의 무료 사용 가능 횟수는 월 1천회 로 제한되므로, 프로젝트에 사용할 분량은 남겨주세요.

## 실습 목차
---

1. **Tavily Search API 발급:** Tavily AI에 가입하고, API Key를 발급 받습니다.

2. **웹 검색 체인 구성:** 발급 받은 Tavily AI를 활용해 웹 데이터를 검색하고, 그 결과를 바탕으로 답변하는 체인을 구성합니다.

## 0. 환경 설정
- 필요한 라이브러리를 불러옵니다.

In [3]:
import os

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.embeddings import OllamaEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

/Users/bkk/프로젝트/yeardream/.venv-1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- Ollama를 통해 Llama3.1 모델을 불러옵니다.

In [4]:
!ollama pull llama3.2:3B

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


## 1. Tavily Search API 발급

- Tavily Search API는 LLM과 RAG에 최적화된 웹 검색 API로써, LangChain 공식 튜토리얼에서도 활용하는 검색 API입니다.

1. 먼저, 아래 링크에 접속한 후 'Sign-in' 버튼을 눌러 로그인 화면으로 이동합니다.
   - https://app.tavily.com/sign-in
2. 그 다음, 여러분의 Google, GitHub 계정과 연동되는 계정을 만들거나 아래의 'Sign up' 버튼을 눌러 새로운 계정을 생성합니다.
3. 계정을 생성한 후, 아래 링크에 접속하여 default API Key를 복사하여 아래 코드에 적용합니다.
   - https://app.tavily.com/home

In [5]:
# Tavily API key는 tvly- 로 시작하는 문자열입니다.
# API Key를 입력했다면, 이 셀을 실행해서 API Key를 환경 변수에 등록합니다.
os.environ["TAVILY_API_KEY"] = "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"

API Key를 성공적으로 등록했다면, 아래 코드를 실행해서 Tavily Search API가 잘 작동하는 지 확인합니다.

In [6]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_search_tool = TavilySearchResults(max_results=5)

/var/folders/m9/22pbfsqj2k52h8fn3krgqw740000gn/T/ipykernel_52611/3375671062.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search_tool = TavilySearchResults(max_results=5)


In [7]:
tavily_search_tool.invoke({"query": "LangChain과 LlamaIndex의 특징과 차이점은 무엇인가요?"})

[{'title': 'LlamaIndex 및 LangChain 비교: 차이점은 무엇인가요? | IBM',
  'url': 'https://www.ibm.com/kr-ko/think/topics/llamaindex-vs-langchain',
  'content': '## LangChain 및 LlamaIndex 비교: 주요 차이점\n\nLlamaIndex와 LangChain은 모두 사용자가 RAG 지원 LLM 애플리케이션을 구축할 수 있도록 허용하지만 프로젝트에 대해 두 가지의 고유한 접근 방식을 제공합니다. LlamaIndex는 관련 정보를 검색하기 위해 데이터베이스를 쿼리할 때 뛰어난 반면, LangChain은 유연성이 더 뛰어나 다양한 사용 사례에 활용할 수 있으며, 특히 모델과 도구를 워크플로에 연결할 때 유용합니다.\n\n### LlamaIndex를 선택해야 하는 경우\n\nLlamaIndex는 가벼운 개발 리프트를 사용하는 간단한 RAG 애플리케이션에 이상적입니다. 의미적 관련성을 기반으로 효율적이고 정확한 데이터 검색에 탁월합니다. 다음과 같은 강점이 있습니다.\n\n검색 앱: 효율적인 데이터 저장과 의미론적 유사성에 기반한 데이터 검색에 중점을 둔 LlamaIndex는 간소화된 RAG 애플리케이션을 위한 좋은 선택입니다. 사용 사례에는 내부 조직 참조 시스템과 지식 관리가 포함됩니다.\n\n속도 및 정밀도: 고급 검색 알고리즘을 통해 LlamaIndex는 높은 수준의 정확도로 효율적인 데이터 검색에 최적화되어 있습니다.\n\n최소화되고 간소화된 앱 개발: LlamaIndex의 집중적인 노력 덕분에 효율적인 앱 제작 프로세스가 가능합니다. 사용자는 최소한의 시간으로 RAG 애플리케이션을 시작하고 실행할 수 있습니다.\n\n계층적 문서: LlamaIndex는 문서 계층 구조가 가장 중요한 기업 내에서 지식 관리 시스템을 구현하는 것처럼 텍스트가 많은 프로젝트에 적합한 선택입니다. [...] # LlamaIndex 및 LangChain 비교: 차이점은 

아래 예시와 비슷한 구조로 출력된다면 정상적으로 API를 적용한 것입니다.
```python
[{'url': 'https://stack-queue.tistory.com/1684',
  'content': 'Langchain은 LlamaIndex보다도 더 유연하며, 사용자가 응용프로그램의 동작을 자유롭게 사용자 정의할 수 있습니다. LlamaIndex는 명확한 검색 및 검색 응용프로그램을 구축하는 데 특별히 설계되었습니다. LLM에 대한 질의 및 관련 문서 검색을 위한 간단한 인터페이스를'},
 {'url': 'https://datasciencedojo.com/blog/llamaindex-vs-langchain/',
  'content': 'In conclusion, by recognizing the unique features and differences between LlamaIndex and LangChain, developers can more effectively align their needs with the capabilities of these tools, resulting in the construction of more efficient, powerful, and accurate search and retrieval applications powered by large language models\nOver 95,000 individuals trust our LinkedIn newsletter for the latest insights in data science, generative AI, and large language models.\n Reviews\nConsulting\nCommunity\nIndex:\nLlamaIndex vs LangChain: Understand the key differences\nMarch 3\nIn the debate of LlamaIndex vs LangChain, developers can align their needs with the capabilities of both tools, resulting in an efficient application.\n Read this blog on LlamaIndex to learn more in detail\nFeatures of LlamaIndex:\nLlamaIndex is an innovative tool designed to enhance the utilization of large language models (LLMs) by seamlessly connecting your data with the powerful computational capabilities of these models. Below are detailed use cases for LlamaIndex, specifically centered around semantic search, and case studies that highlight its indexing capabilities:\nSemantic Search with LlamaIndex:\nCase studies showcasing indexing capabilities:\nDue to its granular control and adaptability, LangChain’s framework is specifically designed to build complex applications, including context-aware query engines. As developers and innovators seek tools to expand the reach of LLMs, delving into the offerings of LLamaIndex and LangChain can guide them toward creating standout applications that resonate with efficiency, accuracy, and creativity.\n'},
 {'url': 'https://www.vellum.ai/blog/llamaindex-vs-langchain-comparison',
  'content': "LlamaIndex doesn't have a sandbox for testing prompts. It offers ready-to-use prompt templates, which are good for basic testing but might be limiting for more complex needs, as some users have noted. LangChain, like Llamaindex, offers simple prompt templates for various uses. It's handy for inspiration and starting out."},
```

API 적용이 잘 되었다면, 이제 이 검색 결과를 활용해 답변을 생성하는 간단한 체인을 구성해 봅시다.

## 2. 웹 검색 체인 구성

웹 검색 기반 답변 체인은 RAG 기반 답변 체인과 비슷한 구조를 사용해 구현할 수 있습니다.

4챕터 실습3에서 저희는 Vector DB에서 사용자의 질문과 관련이 있는 Document를 Retrieve 했고, 그 텍스트를 이어 붙여서 프롬프트에 증강하였습니다.

이번 실습에서 구현할 웹 체인도 이와 비슷하게, 웹 검색 결과를 이어 붙여서 프롬프트에 증강합니다.


llama3.1 모델을 사용하는 ChatOllama 객체와 OllamaEmbeddings 객체를 생성합니다.

In [8]:
llm = ChatOllama(model="llama3.2:3B")
embeddings = OllamaEmbeddings(model="llama3.2:3B")

/var/folders/m9/22pbfsqj2k52h8fn3krgqw740000gn/T/ipykernel_52611/2126658669.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="llama3.2:3B")


다음으로, 사용자의 질문에 대해 Tavily Search API를 통해 여러 문서를 수집하고, 그 결과를 이어 붙이는 함수를 정의합니다.

In [9]:
def tavily_search_and_concat(query: str) -> str:
    results = tavily_search_tool.invoke({"query": query})
    return "\n".join([result["content"] for result in results])

`init_chain()` 함수를 정의합니다.
- 체인에서 `tavily_search_and_concat` 함수를 사용하는 것을 확인할 수 있습니다.

In [10]:
def init_chain():
    messages_with_contexts = [
        ("system", "웹 검색을 통해 수집한 정보를 바탕으로 질문에 답하세요."),
        ("human", "정보: {context}.\n{question}."),
    ]

    prompt_with_context = ChatPromptTemplate.from_messages(messages_with_contexts)

    # 체인 구성
    # Context로 Tavily Serach API 결과를 이어 붙이는 함수를 사용합니다.
    qa_chain = (
        {"context": tavily_search_and_concat, "question": RunnablePassthrough()}
        | prompt_with_context
        | llm
        | StrOutputParser()
    )
    
    return qa_chain

In [11]:
qa_chain = init_chain()

Chain 구성이 완료되었으므로, 사용해봅시다.

In [12]:
question = "LangChain과 LlamaIndex의 특징과 차이점은 무엇인가요?"

print(qa_chain.invoke(question))

LangChain과 LlamaIndex는 사용자가 RAG 지원 LLM 애플리케이션을 구축할 수 있도록 허용하지만, 프로젝트에 대해 두 가지의 고유한 접근 방식을 제공합니다. 

LlamaIndex는 관련 정보를 검색하기 위해 데이터베이스를 쿼리할 때 뛰어난 반면, LangChain은 유연성이 더 뛰어나 다양한 사용 사례에 활용할 수 있으며, 특히 모델과 도구를 워크플로에 연결할 때 유용합니다.

LangChain의 장점은 다음과 같습니다.

다양한 사용 사례: LangChain은 LLM, 도구 및 통합의 샌드박스이며, 사용자는 특정 프로젝트 요구 사항에 맞게 이들을 서로 연결할 수 있습니다.

멀티모달 데이터 소스: LlamaIndex가 이미지와 텍스트를 지원하는 반면, LangChain의 미디어 지원은 훨씬 더 다양합니다. 

세부적인 제어: LangChain의 앱 생성에 대한 단계별 접근 방식은 사용자에게 프로세스의 각 체인의 모든 단계에서 기능을 최대한 제어할 수 있는 권한을 제공합니다.

컨텍스트 유지: 정교한 메모리 관리 능력으로 LangChain에서 생성된 앱은 이전 상호 작용을 참조하고 긴 대화에서도 정확성을 유지할 수 있습니다.

복잡한 쿼리와 데이터 구조: LlamaIndex가 의미적 유사성을 위해 구축된 반면, LangChain은 사용자가 키워드 검색을 추가하는 등의 검색 기술을 결합할 수 있도록 허용합니다. 또한 모듈식 인터페이스, 멀티모달 지원, 다양한 통합 기능으로 복잡한 데이터 구조를 더욱 효과적으로 처리할 수 있습니다.

LangChain의 장점과 LlamaIndex의 특징은 다음과 같습니다:

LangChain의 장점:

* 다양한 사용 사례
* 멀티모달 데이터 소스
* 세부적인 제어
* 컨텍스트 유지
* 복잡한 쿼리 및 데이터 구조

LlamaIndex의 특징:

* 검색 앱: 효율적인 데이터 저장과 의미론적 유사성에 기반한 데이터 검색에 중점을 둔 LlamaIndex는 간소화된 RAG 애플리케이션을 위한 좋은 선택입니다. 
* 

Tavily Search API를 활용해서 저희는 사용자의 질문에 대해 웹에서 정보를 수집해서 이를 바탕으로 답변하는 체인을 구성했습니다. 

Adaptive RAG를 구현하면서 이 체인을 활용해서 웹 검색 기능을 추가해 볼 것입니다.